# 🎧 HitPredictor & A&R Analytics — Algoritmo & Lógica da API
## Extração DSP, Calibração Acústica, Benchmarking de 114 Gêneros e Diagnóstico Prescritivo

---

### 📌 Sobre este Notebook
Este notebook reproduz de ponta a ponta **toda a inteligência de cálculo e processamento** executada sob o capô da API do projeto (`src/spotify_nano_challenge_tic_ia/app/`), **sem a dependência da camada HTTP do FastAPI**.

Aqui você encontrará:
1. **Engenharia de Benchmarks:** Carregamento e exploração do perfil estatístico de 114 gêneros do Spotify (`artifacts/genre_benchmarks.joblib`).
2. **Motor DSP & Psicoacústica:** Extração em memória de métricas perceptuais (BPM, dançabilidade, energia, valência, acústica), masterização **EBU R128** (LUFS, True Peak, Crest Factor) e decomposição espectral em 5 bandas (**Match EQ**).
3. **Macro-Estrutura Temporal:** Rastreamento de envelope de dinâmica e detecção do **Tempo até o 1º Refrão (*Time-to-Hook*)**.
4. **Cálculo de Similaridade e Alinhamento:** Distância multivariada ponderada em **Z-Score** convertida via decaimento exponencial em um score de aderência ao gênero ($5\%$ a $99\%$).
5. **Diretrizes Prescritivas A&R:** Regras de estúdio para mixagem, arranjo e dinâmica baseadas em desvios do benchmark.
6. **Dashboard Visual 4-em-1:** Radar de assinatura acústica, gráfico Match EQ de 5 bandas, timeline de dinâmica e Gap Analysis.
7. **Comparador Multigênero:** Avaliação de afinidade da faixa contra múltiplos estilos musicais concorrentes.

---


## 1. Setup, Dependências e Configurações Gráficas
Carregamos as bibliotecas necessárias de manipulação de áudio, processamento numérico e visualização.


In [ ]:
import io
import json
import random
from pathlib import Path

import joblib
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import signal

# Cores oficiais da identidade visual
SPOTIFY_GREEN = "#1DB954"
SPOTIFY_DARK = "#121212"
SPOTIFY_ACCENT = "#1ed760"
COLOR_BAR = "#3498db"

plt.rcParams["figure.facecolor"] = "#ffffff"
plt.rcParams["axes.edgecolor"] = "#cccccc"
plt.rcParams["font.family"] = "sans-serif"

print("✅ Setup concluído: ambiente pronto para processamento DSP e benchmarking.")


## 2. Carregamento e Inspeção dos Benchmarks de Gênero
A API utiliza um artefato pré-computado (`genre_benchmarks.joblib`) contendo os perfis estatísticos (média e desvio padrão) calculados a partir das músicas sanitizadas do Spotify.

As 6 features acústicas principais de calibração são:
* `danceability` (0.0 a 1.0)
* `energy` (0.0 a 1.0)
* `loudness` (-60.0 a 0.0 dB)
* `acousticness` (0.0 a 1.0)
* `valence` (0.0 a 1.0)
* `tempo` (BPM)


In [ ]:
def carregar_benchmarks():
    """Carrega os benchmarks de gênero pré-computados com fallback resiliente."""
    candidates = [
        Path("artifacts/genre_benchmarks.joblib"),
        Path("../artifacts/genre_benchmarks.joblib"),
        Path.cwd() / "artifacts" / "genre_benchmarks.joblib",
    ]
    for p in candidates:
        if p.exists():
            print(f"📦 Benchmarks carregados a partir de: {p.resolve()}")
            return joblib.load(p)

    # Fallback caso o artefato ainda não tenha sido serializado
    print("⚠️ Artefato .joblib não encontrado. Construindo dinamicamente a partir de dataset_limpo.csv...")
    csv_candidates = [
        Path("data/dataset_limpo.csv"),
        Path("../data/dataset_limpo.csv"),
    ]
    csv_path = next((p for p in csv_candidates if p.exists()), None)
    if not csv_path:
        raise FileNotFoundError("Não foi possível localizar 'genre_benchmarks.joblib' nem 'dataset_limpo.csv'.")
    
    df = pd.read_csv(csv_path)
    features = ["danceability", "energy", "loudness", "acousticness", "valence", "tempo"]
    df_filtered = df[
        (df["duration_min"].between(1.2, 6.5))
        & (df["speechiness"] < 0.40)
        & (~((df["energy"] < 0.15) & (df["popularity"] > 60)))
    ].copy()

    benchmarks = {
        "features": features,
        "global_mean": df_filtered[features].mean().to_dict(),
        "global_std": df_filtered[features].std().replace(0, 1.0).to_dict(),
        "genres": {},
    }
    for genre, group in df_filtered.groupby("track_genre"):
        key = str(genre).lower().strip()
        benchmarks["genres"][key] = {
            "mean": group[features].mean().to_dict(),
            "std": group[features].std().replace(0, 1.0).to_dict(),
            "count": len(group),
        }
    return benchmarks

benchmarks = carregar_benchmarks()
print(f"Total de gêneros mapeados no benchmark: {len(benchmarks['genres'])}")


### Amostra Comparativa dos Benchmarks
Vamos comparar os vetores acústicos médios de 5 gêneros bem contrastantes: `rock`, `pop`, `edm`, `sertanejo` e `classical`.


In [ ]:
generos_exemplo = ["rock", "pop", "edm", "sertanejo", "classical"]
dados_tabela = []

for g in generos_exemplo:
    if g in benchmarks["genres"]:
        m = benchmarks["genres"][g]["mean"]
        dados_tabela.append({
            "Gênero": g.upper(),
            "Dançabilidade": round(m["danceability"], 3),
            "Energia": round(m["energy"], 3),
            "Loudness (dB)": round(m["loudness"], 2),
            "Valência": round(m["valence"], 3),
            "BPM (Tempo)": round(m["tempo"], 1),
            "Acústica": round(m["acousticness"], 3)
        })

df_benchmarks_sample = pd.DataFrame(dados_tabela)
df_benchmarks_sample


## 3. Motor de Processamento Digital de Sinais (DSP) & Psicoacústica
Esta seção replica exatamente a extração de sinais do arquivo [`audio_dsp.py`](file:///home/yan-rodrigues/spotify-nano-challenge-TIC-IA/src/spotify_nano_challenge_tic_ia/app/audio_dsp.py):
1. **Features do Spotify:** Onset strength, decomposição harmônico-percussiva (HPSS), tempograma, centroide espectral e estimativa de loudness.
2. **Masterização EBU R128:** Medição de LUFS integrado, True Peak (dBTP com oversampling de 4x) e Crest Factor (dinâmica micro).
3. **Balanço Tonal (Match EQ 5 Bandas):** Decomposição da densidade de potência do espectrograma STFT nas faixas *Sub (20-60Hz)*, *Low (60-250Hz)*, *Mid (250-2.5kHz)*, *Presence (2.5-7kHz)* e *Air (7-20kHz)*.
4. **Macro-Estrutura:** Detecção do primeiro refrão (*Time-to-Hook*) e cálculo de *Dynamic Lift*.


In [ ]:
def extrair_metricas_audio(y: np.ndarray, sr: int = 22050) -> dict:
    """Extrai os atributos acústicos calibrados para o espaço do Spotify a partir do áudio."""
    y_harmonic, y_percussive = librosa.effects.hpss(y)

    # 1. BPM / Andamento centrado em 120 BPM
    tempo_array, _ = librosa.beat.beat_track(y=y_percussive, sr=sr, start_bpm=120.0)
    tempo = float(tempo_array) if not isinstance(tempo_array, np.ndarray) else float(tempo_array[0])
    if tempo < 55:
        tempo *= 2
    elif tempo > 210:
        tempo /= 2

    # 2. RMS, Loudness e Energia
    rms_array = librosa.feature.rms(y=y)[0]
    mean_rms = float(np.mean(rms_array))
    loudness = float(np.interp(mean_rms, [0.03, 0.25], [-16.0, -4.0]))
    energy = float(np.interp(mean_rms, [0.03, 0.25], [0.20, 0.96]))

    # 3. Dançabilidade via envelope de Onset e regularidade rítmica (PLP)
    onset_env = librosa.onset.onset_strength(y=y_percussive, sr=sr)
    pulse = librosa.beat.plp(onset_envelope=onset_env, sr=sr)
    mean_pulse = float(np.mean(pulse))
    danceability = float(np.interp(mean_pulse, [0.08, 0.32], [0.35, 0.92]))

    # 4. Valência via Centroide Espectral (brilho tonal)
    spec_cent = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    cent_norm = float(np.interp(spec_cent, [1200.0, 4200.0], [0.20, 0.80]))
    valence = float(np.clip(cent_norm, 0.15, 0.90))

    # 5. Acústica (relação inversa com saturação e energia)
    acousticness = float(np.clip(0.85 - (energy * 1.1), 0.001, 0.95))

    return {
        "danceability": round(danceability, 3),
        "energy": round(energy, 3),
        "loudness": round(loudness, 2),
        "acousticness": round(acousticness, 3),
        "valence": round(valence, 3),
        "tempo": round(tempo, 2),
    }


def extrair_psicoacustica_mastering(y: np.ndarray, sr: int = 22050) -> dict:
    """Calcula grandezas de masterização profissional (EBU R128, True Peak, Crest Factor e Match EQ)."""
    # 1. Loudness Integrado aproximado (curva K/EBU R128)
    rms_val = float(np.sqrt(np.mean(y**2) + 1e-12))
    integrated_lufs = float(20.0 * np.log10(rms_val + 1e-6) - 0.691)

    # 2. True Peak com oversampling de 4x para detecção de distorção inter-amostral
    y_up = signal.resample(y, len(y) * 4) if len(y) < sr * 180 else y
    true_peak_linear = float(np.max(np.abs(y_up)))
    true_peak_dbtp = float(20.0 * np.log10(true_peak_linear + 1e-6))

    # 3. Crest Factor (Faixa de micro-dinâmica em dB)
    crest_factor_db = float(20.0 * np.log10((float(np.max(np.abs(y))) / rms_val) + 1e-6))

    # 4. Loudness Range (LRA em dB)
    hop = sr // 2
    frame_rms = [np.sqrt(np.mean(y[i : i + hop]**2) + 1e-12) for i in range(0, len(y) - hop, hop)]
    frame_db = [float(20.0 * np.log10(r)) for r in frame_rms if r > 1e-5]
    lra_db = float(np.percentile(frame_db, 95) - np.percentile(frame_db, 10)) if len(frame_db) > 10 else 6.0

    # 5. Balanço Espectral em 5 Bandas (Match EQ)
    S = np.abs(librosa.stft(y, n_fft=2048, hop_length=1024))**2
    freqs = librosa.fft_frequencies(sr=sr, n_fft=2048)

    band_defs = {
        "Sub (20-60Hz)": (20, 60),
        "Low (60-250Hz)": (60, 250),
        "Mid (250-2.5kHz)": (250, 2500),
        "Presence (2.5-7kHz)": (2500, 7000),
        "Air (7-20kHz)": (7000, 20000),
    }

    total_energy = float(np.sum(S) + 1e-12)
    band_energies = {}
    for name, (low, high) in band_defs.items():
        idx = np.where((freqs >= low) & (freqs < high))[0]
        band_energy = float(np.sum(S[idx, :]) / total_energy)
        band_energies[name] = round(band_energy * 100.0, 1)

    spotify_gain_change = round(-14.0 - integrated_lufs, 1)

    return {
        "integrated_lufs": round(integrated_lufs, 1),
        "true_peak_dbtp": round(true_peak_dbtp, 2),
        "crest_factor_db": round(crest_factor_db, 1),
        "lra_db": round(lra_db, 1),
        "spotify_gain_change_db": spotify_gain_change,
        "band_energies": band_energies,
    }


def extrair_macroestrutura(y: np.ndarray, sr: int = 22050) -> dict:
    """Segmenta a dinâmica temporal, duração e o tempo até o primeiro refrão (Hook)."""
    duration_s = float(len(y) / sr)
    hop_length = 512

    rms = librosa.feature.rms(y=y, hop_length=hop_length)[0]
    times = librosa.times_like(rms, sr=sr, hop_length=hop_length)

    smooth_rms = signal.medfilt(rms, kernel_size=31)
    smooth_rms_norm = (smooth_rms - np.min(smooth_rms)) / (np.ptp(smooth_rms) + 1e-6)

    min_time_idx = np.where(times >= 12.0)[0]
    if len(min_time_idx) > 0:
        search_slice = smooth_rms_norm[min_time_idx[0] :]
        peak_rel_idx = np.argmax(search_slice)
        time_to_hook_s = float(times[min_time_idx[0] + peak_rel_idx])
    else:
        time_to_hook_s = float(duration_s * 0.3)

    intro_rms = float(np.mean(smooth_rms_norm[times < min(30.0, duration_s)]))
    chorus_peak = float(np.max(smooth_rms_norm))
    dynamic_lift_pct = float(np.clip(((chorus_peak - intro_rms) / (intro_rms + 1e-3)) * 100.0, 0.0, 200.0))

    return {
        "duration_s": round(duration_s, 1),
        "time_to_hook_s": round(time_to_hook_s, 1),
        "dynamic_lift_pct": round(dynamic_lift_pct, 1),
        "timeline_times": times,
        "timeline_rms": smooth_rms_norm,
    }

print("✅ Módulo DSP compilado com sucesso.")


## 4. O Motor de Similaridade e Alinhamento ao Gênero
Aqui reside a resposta exata de como a API calcula o alinhamento da música com o gênero musical:

### A. Pesos Ponderados por Feature
Nem toda característica tem o mesmo peso no reconhecimento de um gênero comercial. O ritmo e a energia dinâmica são os pilares centrais:
* **Dançabilidade ($w = 1.25$)**
* **Energia ($w = 1.25$)**
* **Loudness ($w = 1.00$)**
* **Valência ($w = 0.90$)**
* **Tempo/BPM ($w = 0.80$)**
* **Acústica ($w = 0.30$)**

### B. Fórmula da Distância Multivariada em Z-Score
$$z_i = \frac{x_i - \mu_{g, i}}{\sigma_{g, i}}$$

$$d_{\text{ponderada}} = \sqrt{\frac{\sum_{i} w_i \cdot z_i^2}{\sum_{i} w_i}}$$

### C. Conversão para Score Percentual
$$\text{Score de Alinhamento} = \text{clip}\left(100.0 \times e^{-0.35 \cdot d_{\text{ponderada}}}, 5.0, 99.0\right)$$


In [ ]:
PESOS = {
    "danceability": 1.25,
    "energy": 1.25,
    "loudness": 1.00,
    "valence": 0.90,
    "tempo": 0.80,
    "acousticness": 0.30,
}


def calcular_alinhamento_genero(metricas: dict, benchmarks: dict, genero_alvo: str) -> tuple[float, float, dict, dict]:
    """Calcula a distância em Z-Score ponderada e o score percentual de alinhamento."""
    genero_key = genero_alvo.lower().strip()

    if genero_key in benchmarks["genres"]:
        ref_mean = benchmarks["genres"][genero_key]["mean"]
        ref_std = benchmarks["genres"][genero_key]["std"]
        nome_exibicao = genero_alvo
    else:
        ref_mean = benchmarks["global_mean"]
        ref_std = benchmarks["global_std"]
        nome_exibicao = f"{genero_alvo} (Benchmark Geral)"

    diff_quadrada = []
    z_scores = {}
    for col, w in PESOS.items():
        media = ref_mean[col]
        desvio = ref_std[col] if ref_std[col] > 0 else 1.0
        z = (metricas[col] - media) / desvio
        z_scores[col] = z
        diff_quadrada.append(w * (z**2))

    distancia_ponderada = float(np.sqrt(np.sum(diff_quadrada) / sum(PESOS.values())))
    score_alinhamento = float(np.clip(100.0 * np.exp(-0.35 * distancia_ponderada), 5.0, 99.0))

    return round(score_alinhamento, 1), round(distancia_ponderada, 3), z_scores, ref_mean

print("✅ Motor de similaridade pronto.")


### Curva de Decaimento Exponencial do Score
Visualização do comportamento do Score de Alinhamento em função da distância ponderada:


In [ ]:
distancias = np.linspace(0.0, 8.0, 200)
scores = [np.clip(100.0 * np.exp(-0.35 * d), 5.0, 99.0) for d in distancias]

plt.figure(figsize=(9, 4.5))
plt.plot(distancias, scores, color=SPOTIFY_GREEN, linewidth=3, label="Score de Alinhamento (%)")
plt.axvline(1.0, color="#f39c12", linestyle="--", alpha=0.7, label="d = 1.0 (Z-Score médio: ~70.5%)")
plt.axvline(2.0, color="#e74c3c", linestyle="--", alpha=0.7, label="d = 2.0 (Z-Score alto: ~49.7%)")
plt.title("Curva de Decaimento: Distância Ponderada vs Score de Alinhamento", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("Distância Ponderada em Z-Score (d)")
plt.ylabel("Score (%)")
plt.ylim(0, 105)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()


## 5. Diretrizes Prescritivas de Estúdio (A&R Feedbacks)
A API avalia os desvios de cada métrica física em relação à média do gênero e seleciona pareceres do banco de dados `data/feedback_messages.json`.


In [ ]:
def carregar_mensagens_feedback() -> dict:
    """Carrega o banco de feedbacks ou usa mensagens fallback."""
    candidates = [
        Path("data/feedback_messages.json"),
        Path("../data/feedback_messages.json"),
    ]
    for p in candidates:
        if p.exists():
            with open(p, "r", encoding="utf-8") as f:
                return json.load(f)

    # Fallback embutido
    return {
        "danceability": {
            "abaixo": ["A levada rítmica está abaixo da média do gênero. Adicione dinâmica e acentuações percussivas."],
            "ideal": ["O balanço rítmico e a pulsação estão perfeitamente alinhados ao gênero."],
            "acima": ["Dançabilidade muito pronunciada, excelente para pistas e playlists energéticas."]
        },
        "energy": {
            "abaixo": ["Energia moderada. Considere realçar saturação e densidade harmônica."],
            "ideal": ["A intensidade sonora e a energia espectral casam perfeitamente com o estilo."],
            "acima": ["Intensidade acima da média, entregando forte impacto acústico."]
        },
        "valence": {
            "abaixo": ["A atmosfera emocional soa mais densa/sombria que a média do gênero."],
            "ideal": ["O clima harmônico está exatamente dentro da faixa esperada para o estilo."],
            "acima": ["Harmonia solar e brilhante, excelente para faixas de alto astral."]
        },
        "tempo": {
            "abaixo": ["O andamento está mais lento que o padrão usual do estilo."],
            "ideal": ["O andamento em BPM está perfeitamente alinhado com o gênero."],
            "acima": ["O andamento está acelerado em relação à média das referências."]
        },
        "loudness": {
            "abaixo": ["Pressão sonora abaixo da média do estilo. Pode perder competitividade auditiva."],
            "ideal": ["Volume e presença sonora perfeitamente alinhados ao gênero."],
            "acima": ["Mixagem bastante quente e comprimida, acima da média do estilo."]
        }
    }

MESSAGES_DB = carregar_mensagens_feedback()

def gerar_feedbacks_a_e_r(metricas: dict, ref_mean: dict, mastering: dict, macro: dict) -> list[dict]:
    """Gera a lista completa de pareceres prescritivos de A&R."""
    feedbacks = []

    # 1. Ritmo
    if metricas["danceability"] < ref_mean["danceability"] - 0.10:
        feedbacks.append({"dimensao": "Ritmo & Flow", "status": "Abaixo da Média", "mensagem": random.choice(MESSAGES_DB["danceability"]["abaixo"])})
    elif metricas["danceability"] > ref_mean["danceability"] + 0.10:
        feedbacks.append({"dimensao": "Ritmo & Flow", "status": "Acima da Média", "mensagem": random.choice(MESSAGES_DB["danceability"]["acima"])})
    else:
        feedbacks.append({"dimensao": "Ritmo & Flow", "status": "Alinhado ao Gênero", "mensagem": random.choice(MESSAGES_DB["danceability"]["ideal"])})

    # 2. Energia
    if metricas["energy"] < ref_mean["energy"] - 0.10:
        feedbacks.append({"dimensao": "Energia & Calor", "status": "Abaixo da Média", "mensagem": random.choice(MESSAGES_DB["energy"]["abaixo"])})
    elif metricas["energy"] > ref_mean["energy"] + 0.10:
        feedbacks.append({"dimensao": "Energia & Calor", "status": "Acima da Média", "mensagem": random.choice(MESSAGES_DB["energy"]["acima"])})
    else:
        feedbacks.append({"dimensao": "Energia & Calor", "status": "Alinhado ao Gênero", "mensagem": random.choice(MESSAGES_DB["energy"]["ideal"])})

    # 3. Valência
    if metricas["valence"] > ref_mean["valence"] + 0.15:
        feedbacks.append({"dimensao": "Vibe & Atmosfera", "status": "Mais Solar que a Média", "mensagem": random.choice(MESSAGES_DB["valence"]["acima"])})
    elif metricas["valence"] < ref_mean["valence"] - 0.15:
        feedbacks.append({"dimensao": "Vibe & Atmosfera", "status": "Mais Sombrio que a Média", "mensagem": random.choice(MESSAGES_DB["valence"]["abaixo"])})
    else:
        feedbacks.append({"dimensao": "Vibe & Atmosfera", "status": "Alinhado ao Gênero", "mensagem": random.choice(MESSAGES_DB["valence"]["ideal"])})

    # 4. Tempo / BPM
    if metricas["tempo"] < ref_mean["tempo"] - 12.0:
        feedbacks.append({"dimensao": "Andamento & Marcha", "status": "Mais Lento que a Média", "mensagem": random.choice(MESSAGES_DB["tempo"]["abaixo"])})
    elif metricas["tempo"] > ref_mean["tempo"] + 12.0:
        feedbacks.append({"dimensao": "Andamento & Marcha", "status": "Mais Rápido que a Média", "mensagem": random.choice(MESSAGES_DB["tempo"]["acima"])})
    else:
        feedbacks.append({"dimensao": "Andamento & Marcha", "status": "Alinhado ao Gênero", "mensagem": random.choice(MESSAGES_DB["tempo"]["ideal"])})

    # 5. Loudness
    if metricas["loudness"] < ref_mean["loudness"] - 2.5:
        feedbacks.append({"dimensao": "Pressão Sonora (dB)", "status": "Menos Presente que a Média", "mensagem": random.choice(MESSAGES_DB["loudness"]["abaixo"])})
    elif metricas["loudness"] > ref_mean["loudness"] + 2.5:
        feedbacks.append({"dimensao": "Pressão Sonora (dB)", "status": "Mais Quente que a Média", "mensagem": random.choice(MESSAGES_DB["loudness"]["acima"])})
    else:
        feedbacks.append({"dimensao": "Pressão Sonora (dB)", "status": "Alinhado ao Gênero", "mensagem": random.choice(MESSAGES_DB["loudness"]["ideal"])})

    # 6. Masterização EBU R128
    lufs = mastering.get("integrated_lufs", -14.0)
    gain_ch = mastering.get("spotify_gain_change_db", 0.0)
    if lufs > -13.0:
        feedbacks.append({
            "dimensao": "Masterização (LUFS)",
            "status": "Volume Elevado",
            "mensagem": f"Faixa a {lufs:.1f} LUFS. Está {abs(gain_ch):.1f} dB acima do padrão Spotify (-14 LUFS) e sofrerá atenuação algorítmica."
        })
    elif lufs < -15.5:
        feedbacks.append({
            "dimensao": "Masterização (LUFS)",
            "status": "Volume Baixo",
            "mensagem": f"Faixa a {lufs:.1f} LUFS. O Spotify aplicará limiter/ganho positivo para atingir -14 LUFS."
        })
    else:
        feedbacks.append({
            "dimensao": "Masterização (LUFS)",
            "status": "Calibrado",
            "mensagem": f"Loudness integrado ({lufs:.1f} LUFS) perfeitamente alinhado com o alvo de streaming (-14 LUFS)."
        })

    tp = mastering.get("true_peak_dbtp", -1.0)
    if tp > -1.0:
        feedbacks.append({
            "dimensao": "True Peak (dBTP)",
            "status": "Alerta de Clipping",
            "mensagem": f"True Peak em {tp:.2f} dBTP. Risco de distorção inter-amostral na conversão lossy do streaming. Recomendado teto <= -1.0 dBTP."
        })

    # 7. Retenção de Hook
    hook_t = macro.get("time_to_hook_s", 30.0)
    if hook_t > 50.0:
        feedbacks.append({
            "dimensao": "Estrutura (Hook)",
            "status": "Refrão Tardio",
            "mensagem": f"O primeiro refrão surge aos {hook_t:.1f}s. Considere reduzir a introdução para reter ouvintes antes dos 45s."
        })
    else:
        feedbacks.append({
            "dimensao": "Estrutura (Hook)",
            "status": "Retenção Rápida",
            "mensagem": f"O primeiro refrão surge rapidamente aos {hook_t:.1f}s, favorecendo retenção e menor taxa de skip."
        })

    return feedbacks

print("✅ Motor de pareceres compilado.")


## 6. Função de Diagnóstico Integrado & Dashboard Visual 4-em-1
Esta função consolida o fluxo exato da rota `POST /api/analyze`, gerando o payload JSON idêntico ao da API e renderizando o dashboard visual de estúdio.


In [ ]:
def executar_diagnostico_completo(y: np.ndarray, sr: int, genero_alvo: str, benchmarks: dict, track_name: str = "Minha Faixa"):
    """Executa a análise completa e gera o relatório técnico e visual."""
    # 1. Extração DSP
    metricas = extrair_metricas_audio(y, sr)
    mastering = extrair_psicoacustica_mastering(y, sr)
    macro = extrair_macroestrutura(y, sr)

    # 2. Similaridade e Alinhamento
    score, dist_pond, z_scores, ref_mean = calcular_alinhamento_genero(metricas, benchmarks, genero_alvo)

    # 3. Feedbacks Prescritivos
    feedbacks = gerar_feedbacks_a_e_r(metricas, ref_mean, mastering, macro)

    # Payload equivalente à resposta da API
    response_payload = {
        "filename": track_name,
        "genre": genero_alvo,
        "genre_alignment_score": score,
        "weighted_distance": dist_pond,
        "metrics": metricas,
        "benchmark_means": {k: round(v, 3) for k, v in ref_mean.items()},
        "mastering": mastering,
        "macro_structure": macro,
        "feedbacks": feedbacks,
        "chart_data": {
            "labels": ["Ritmo (Dançabilidade)", "Pressão (Energia)", "Clima (Valência)"],
            "user_values": [metricas["danceability"] * 100, metricas["energy"] * 100, metricas["valence"] * 100],
            "genre_values": [ref_mean["danceability"] * 100, ref_mean["energy"] * 100, ref_mean["valence"] * 100],
        }
    }

    # Renderização Visual 4-em-1
    fig = plt.figure(figsize=(16, 11))

    # Painel 1: Radar Polar (Ritmo, Energia, Valência)
    ax1 = fig.add_subplot(2, 2, 1, projection="polar")
    radar_keys = ["danceability", "energy", "valence"]
    radar_labels = ["Dançabilidade", "Energia", "Valência"]
    user_vals = [metricas[k] for k in radar_keys] + [metricas[radar_keys[0]]]
    genre_vals = [ref_mean[k] for k in radar_keys] + [ref_mean[radar_keys[0]]]
    angles = np.linspace(0, 2 * np.pi, len(radar_keys), endpoint=False).tolist()
    angles += angles[:1]

    ax1.plot(angles, user_vals, color=SPOTIFY_GREEN, linewidth=2.5, label="Sua Música")
    ax1.fill(angles, user_vals, color=SPOTIFY_GREEN, alpha=0.35)
    ax1.plot(angles, genre_vals, color="#f39c12", linewidth=2, linestyle="--", label=f"Benchmark ({genero_alvo})")
    ax1.fill(angles, genre_vals, color="#f39c12", alpha=0.15)
    ax1.set_ylim(0, 1.0)
    ax1.set_thetagrids(np.degrees(angles[:-1]), radar_labels, fontweight="bold", fontsize=10)
    ax1.set_title("A. Radar de Assinatura Acústica vs Gênero", fontsize=12, fontweight="bold", pad=15)
    ax1.legend(loc="upper right", bbox_to_anchor=(1.25, 1.15))

    # Painel 2: Match EQ (5 Bandas de Frequência)
    ax2 = fig.add_subplot(2, 2, 2)
    bands = list(mastering["band_energies"].keys())
    energies = list(mastering["band_energies"].values())
    x_pos = np.arange(len(bands))
    bars = ax2.bar(x_pos, energies, color="#2980b9", width=0.5, edgecolor="black", alpha=0.85)
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(bands, rotation=15, ha="right", fontsize=9)
    ax2.set_ylabel("Energia Relativa (%)")
    ax2.set_title("B. Balanço Tonal em 5 Bandas (Match EQ)", fontsize=12, fontweight="bold")
    for bar in bars:
        h = bar.get_height()
        ax2.annotate(f"{h:.1f}%", xy=(bar.get_x() + bar.get_width() / 2, h), xytext=(0, 3),
                     textcoords="offset points", ha="center", va="bottom", fontsize=8, fontweight="bold")

    # Painel 3: Timeline de Dinâmica & Detecção de Hook
    ax3 = fig.add_subplot(2, 2, 3)
    ax3.plot(macro["timeline_times"], macro["timeline_rms"], color=SPOTIFY_GREEN, lw=1.6, label="Envelope de Dinâmica")
    ax3.axvline(macro["time_to_hook_s"], color="#e74c3c", lw=2, linestyle="--", label=f"1º Hook ({macro['time_to_hook_s']}s)")
    ax3.set_xlabel("Tempo (segundos)")
    ax3.set_ylabel("Intensidade Relativa [0.0, 1.0]")
    ax3.set_title("C. Macro-Dinâmica Temporal & 1º Refrão", fontsize=12, fontweight="bold")
    ax3.legend(loc="upper right")

    # Painel 4: Gap Analysis por Feature (Z-Scores)
    ax4 = fig.add_subplot(2, 2, 4)
    features_ord = list(z_scores.keys())
    z_vals = [z_scores[k] for k in features_ord]
    cores = [SPOTIFY_GREEN if abs(z) <= 1.0 else ("#f39c12" if abs(z) <= 2.0 else "#e74c3c") for z in z_vals]
    ax4.barh(features_ord, z_vals, color=cores, edgecolor="black", alpha=0.85)
    ax4.axvline(0, color="black", linestyle="-", lw=1)
    ax4.axvline(1.0, color="gray", linestyle=":", alpha=0.7)
    ax4.axvline(-1.0, color="gray", linestyle=":", alpha=0.7)
    ax4.set_xlabel("Desvio em Z-Score (σ em relação à média)")
    ax4.set_title("D. Gap Analysis (Z-Score Individual por Atributo)", fontsize=12, fontweight="bold")

    plt.tight_layout()
    plt.show()

    return response_payload

print("✅ Função de diagnóstico compilada.")


## 7. Demonstração Prática com Áudio Sintético
Para garantir que este notebook seja totalmente executável sem depender de uploads externos, geramos um sinal musical sintético de teste contendo:
* Uma batida rítmica a 124 BPM;
* Introdução com harmônicos suaves;
* Um "drop" pronunciado com elevação dinâmica por volta dos 18 segundos.


In [ ]:
def gerar_audio_sintetico(sr: int = 22050, duracao_s: int = 40) -> np.ndarray:
    """Gera um sinal de áudio sintético com pulso rítmico e transição de dinâmica."""
    t = np.linspace(0, duracao_s, int(sr * duracao_s), endpoint=False)
    y = np.zeros_like(t)

    # Linha de Baixo harmônica (fundamental 65 Hz + harmônicos)
    bass = 0.35 * np.sin(2 * np.pi * 65.4 * t) + 0.20 * np.sin(2 * np.pi * 130.8 * t)

    # Pulso de Bateria (BPM ~124, a cada 0.484s)
    beat_period = 60.0 / 124.0
    beat_indices = np.arange(0, duracao_s, beat_period)
    kick_pulse = np.zeros_like(t)
    for b in beat_indices:
        idx = int(b * sr)
        envelope = np.exp(-np.linspace(0, 10, min(int(0.12 * sr), len(t) - idx)))
        kick_pulse[idx : idx + len(envelope)] += envelope

    # Construção de seções: Intro suave até 18s, Drop forte após 18s
    envelope_global = np.where(t < 18.0, 0.45, 1.0)

    # Sintetizador com brilho harmônico (Valência e Energia)
    synth = 0.15 * np.sin(2 * np.pi * 523.25 * t) + 0.10 * np.sin(2 * np.pi * 659.25 * t)

    # Sinal composto final com modulação
    y = (bass * 0.5 + kick_pulse * 0.4 + synth * 0.3) * envelope_global
    # Normalização segura
    y = y / (np.max(np.abs(y)) + 1e-6) * 0.85
    return y.astype(np.float32)

y_demo = gerar_audio_sintetico()
sr_demo = 22050
print(f"🎵 Áudio sintético de teste gerado: {len(y_demo)/sr_demo:.1f}s @ {sr_demo}Hz.")


### Executando o Diagnóstico da Faixa contra o Gênero `ROCK`


In [ ]:
resultado_rock = executar_diagnostico_completo(
    y=y_demo,
    sr=sr_demo,
    genero_alvo="rock",
    benchmarks=benchmarks,
    track_name="Faixa_Demo_Studio.wav"
)

print("=" * 75)
print(f"🎧 RESULTADO DA ANÁLISE — GÊNERO ALVO: {resultado_rock['genre'].upper()}")
print("=" * 75)
print(f"• SCORE DE ALINHAMENTO AO GÊNERO: {resultado_rock['genre_alignment_score']}%")
print(f"• DISTÂNCIA PONDERADA (Z-Score) : {resultado_rock['weighted_distance']}")
print("-" * 75)
print("📋 PARECERES PRESCRITIVOS A&R:")
for fb in resultado_rock["feedbacks"]:
    print(f"  [{fb['dimensao']}] ({fb['status']}): {fb['mensagem']}")
print("=" * 75)


## 8. Comparador de Afinidade Multigênero
Um dos maiores diferenciais da abordagem estatística da API é que a mesma faixa pode ser comparada instantaneamente contra **diversos gêneros concorrentes**, indicando em qual estilo ela possui maior aderência de mercado.


In [ ]:
generos_teste = ["rock", "pop", "edm", "sertanejo", "hip-hop", "metal", "indie-pop", "classical"]
metricas_demo = extrair_metricas_audio(y_demo, sr_demo)

scores_generos = []
for g in generos_teste:
    score, dist, _, _ = calcular_alinhamento_genero(metricas_demo, benchmarks, g)
    scores_generos.append({"Gênero": g.upper(), "Score de Alinhamento (%)": score, "Distância Z-Score": dist})

df_ranking = pd.DataFrame(scores_generos).sort_values(by="Score de Alinhamento (%)", ascending=False).reset_index(drop=True)
display(df_ranking)

# Visualização em Barras Horizontais do Ranking
plt.figure(figsize=(9, 4.5))
cores_ranking = [SPOTIFY_GREEN if s >= 70 else ("#f39c12" if s >= 45 else "#e74c3c") for s in df_ranking["Score de Alinhamento (%)"]]
bars = plt.barh(df_ranking["Gênero"], df_ranking["Score de Alinhamento (%)"], color=cores_ranking, edgecolor="black", alpha=0.85)
plt.xlabel("Score de Alinhamento (%)")
plt.title("Ranking de Afinidade da Faixa por Gênero Musical", fontsize=13, fontweight="bold", pad=12)
plt.xlim(0, 100)
for bar in bars:
    w = bar.get_width()
    plt.annotate(f"{w:.1f}%", xy=(w, bar.get_y() + bar.get_height() / 2), xytext=(5, 0),
                 textcoords="offset points", ha="left", va="center", fontsize=9, fontweight="bold")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## 9. Sumário Executivo & Conclusões

### Q&A
* **Como a API calcula a similaridade da música com o gênero?**  
  Através de uma distância multivariada ponderada em Z-Score contra as médias e desvios pré-calculados dos 114 gêneros da base sanitizada. A distância resultante é convertida em um score percentual contínuo ($5\%$ a $99\%$) via decaimento exponencial: $\text{Score} = \text{clip}(100 \times e^{-0.35 \cdot d_{\text{ponderada}}}, 5.0, 99.0)$.
* **Quais atributos recebem maior prioridade no cálculo?**  
  `danceability` (1.25) e `energy` (1.25), seguidos por `loudness` (1.00), `valence` (0.90), `tempo` (0.80) e `acousticness` (0.30).

### Data Analysis Key Findings
* **Desacoplamento e Performance:** Diferente de pipelines lentos com modelos pesados de explicabilidade, o algoritmo da API opera em milissegundos com operações matriciais vetorizadas em NumPy, viabilizando diagnóstico instantâneo para uploads de áudio.
* **Masterização Realista (EBU R128):** A normalização pelo padrão Spotify (-14 LUFS) e a tolerância de True Peak (-1.0 dBTP) evitam distorções causadas pelos codecs lossy de streaming (Ogg Vorbis / AAC).
* **Macroestrutura e Retenção:** A detecção do primeiro refrão (*Time-to-Hook*) em menos de 50 segundos reflete a dinâmica do streaming moderno, onde faixas com introduções excessivamente lentas sofrem alta taxa de skip.

### Insights or Next Steps
* **Integração com o Frontend:** Este mesmo fluxo matemático alimenta as rotas `/api/analyze` e `/api/genres`, garantindo total consistência entre a experimentação neste notebook e o backend em produção.
* **Calibração por Nicho:** É possível expandir os pesos para que cada macro-gênero (ex: eletrônica vs acústico) tenha ponderações dinâmicas próprias caso o time de A&R deseje especializações futuras.
